In [1]:
from typing import Any, Optional

from arcengine import FrameDataRaw

from arc import MyArcSession
from arc_agi import OperationMode
from arc_agi import Arcade
import logging
import numpy as np
from dotenv import load_dotenv
from PIL import Image
from sandbox import SandboxOrchestrator
from google.genai import types
from google import genai
from tools import TOOL_LIST, SYSTEM_PROMPT, TAKE_ACTION
from agent import JackAgent

load_dotenv()


PALETTE = np.array(
    [
        [0xFF, 0xFF, 0xFF],
        [0xCC, 0xCC, 0xCC],
        [0x99, 0x99, 0x99],
        [0x66, 0x66, 0x66],
        [0x33, 0x33, 0x33],
        [0x00, 0x00, 0x00],
        [0xE5, 0x3A, 0xA3],
        [0xFF, 0x7B, 0xCC],
        [0xF9, 0x3C, 0x31],
        [0x1E, 0x93, 0xFF],
        [0x88, 0xD8, 0xF1],
        [0xFF, 0xDC, 0x00],
        [0xFF, 0x85, 0x1B],
        [0x92, 0x12, 0x31],
        [0x4F, 0xCC, 0x30],
        [0xA3, 0x56, 0xD6],
    ],
    dtype=np.uint8,
)


def render_frame(frame: np.ndarray) -> Image.Image:
    rgb = PALETTE[np.clip(np.asarray(frame, dtype=np.uint8), 0, 15)]
    return Image.fromarray(rgb).resize((512, 512), Image.NEAREST)


logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S"
)


def current_state_prompt(obs: FrameDataRaw) -> str:
    return f"""Currently playing game: {obs.game_id}. On level #{obs.levels_completed} out of #{obs.win_levels}.

"""


def starting_state_prompt(obs: FrameDataRaw) -> str:
    res = f"""Starting the following game: {obs.game_id}

"""

    return (
        f"New game started. Initial state:\n{json.dumps(obs, indent=2)}\n\n"
        f"The full grid is in /home/agent/state.json. "
        f"Start by calling render_board to see the board, then take actions to explore."
    )

In [2]:

arcade = Arcade(operation_mode=OperationMode("normal"))
scorecard_id: str = arcade.open_scorecard(tags=["jackagent"])
arc_session = MyArcSession(
    game_id=["ls20", "ft09"][0], arcade=arcade, scorecard_id=scorecard_id
)
print(arc_session.obs)
sbx = SandboxOrchestrator()

print(sbx.bash("echo sandbox ready"))
print(sbx.bash("python3 -c 'print(1+122)'"))

agent = JackAgent(sbx=sbx, arc_session=arc_session)

print(agent.contents)

agent.contents = []



2026-04-13 00:50:27 | INFO | Successfully fetched 25 environment(s) from API


00:50:27 Successfully fetched 25 environment(s) from API


2026-04-13 00:50:27 | INFO | Created new scorecard: 9d82fc1f-9c4c-4b0f-b331-5a1db2fadce0


00:50:27 Created new scorecard: 9d82fc1f-9c4c-4b0f-b331-5a1db2fadce0


2026-04-13 00:50:27 | INFO | Successfully fetched metadata for game ls20


00:50:27 Successfully fetched metadata for game ls20


2026-04-13 00:50:27 | INFO | Successfully loaded game class Ls20 from environment_files/ls20/9607627b/ls20.py


00:50:27 Successfully loaded game class Ls20 from environment_files/ls20/9607627b/ls20.py


{
  "game_id": "ls20-9607627b",
  "state": "NOT_FINISHED",
  "levels_completed": 0,
  "win_levels": 7,
  "action_input": {
    "id": 0,
    "data": {},
    "reasoning": null
  },
  "guid": "5b86784b-4282-4669-b04f-9b9558984fda",
  "full_reset": true,
  "available_actions": [
    1,
    2,
    3,
    4
  ]
}
sandbox ready
123
[]


In [3]:
agent

In [2]:
agent.contents = [
    types.Content(role="user", parts=[types.Part(
        text=f"This is the current board. {arc_session.obs}")])
]

agent.contents = [
    types.Content(role="user", parts=[types.Part(text="move up a space")])
]
for _ in range(4):
    agent.generate_response()

NameError: name 'types' is not defined